In [1]:
# inference_and_save.py

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from keras.utils import normalize
from raw_preprocessing import preprocess_raw

# === PARAMETERS ===
MODEL_PATH    = "complex_unet_enhancedv5.keras"       # or .h5
TEST_IMG_PATH = "testimages/img146.jpg"
OUTPUT_DIR    = "testmaskpredict"
SIZE          = 496
N_CLASSES     = 12

# === 1. Load & compile model ===
model = load_model(MODEL_PATH, compile=False)
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# === 2. Prepare output directory ===
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === 3. Preprocess the test image ===
img = preprocess_raw(TEST_IMG_PATH,
                     target_size=SIZE,
                     normalize=True)   # use the updated parameter name

# Add batch dim → (1, H, W, 1)
X = np.expand_dims(img, axis=0)

# Exactly match your training normalization
X = normalize(X, axis=1)

# === 4. Predict & decode mask ===
pred = model.predict(X)[0]               # (H, W, 12)
mask = np.argmax(pred, axis=-1).astype(np.uint8)  # (H, W)

# === 5. Plot & save exactly like your notebook ===
fig, ax = plt.subplots(1, 1,
                       figsize=(4, 4),
                       dpi=124)  # 496px / 4in = 124dpi
ax.imshow(mask,
          cmap="jet",
          vmin=0,
          vmax=N_CLASSES-1,
          interpolation="nearest",
          aspect="equal")
ax.axis("off")
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

base = os.path.splitext(os.path.basename(TEST_IMG_PATH))[0]
out_path = os.path.join(OUTPUT_DIR, f"{base}_jetmask.png")
fig.savefig(out_path,
            dpi=124,
            pad_inches=0,
            bbox_inches="tight")
plt.close(fig)

print(f"✅ Saved mask to {out_path}")


2025-09-04 15:36:15.671669: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-09-04 15:36:15.671696: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-09-04 15:36:15.671700: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-09-04 15:36:15.671716: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-09-04 15:36:15.671727: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-09-04 15:36:19.028117: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
✅ Saved mask to testmaskpredict/img146_jetmask.png


In [2]:
import cv2
import numpy as np

MASK_PATH = "testmaskpredict/img146_jetmask.png"
mask = cv2.imread(MASK_PATH, cv2.IMREAD_GRAYSCALE)
print("Mask shape:", mask.shape)
print("Unique values in mask:", np.unique(mask))


Mask shape: (496, 496)
Unique values in mask: [ 14  26  37  61  69 115 126 173 176 193 207 221]


In [3]:
import os
import numpy as np
import pandas as pd
import cv2

from tensorflow.keras.models import load_model
from keras.utils import normalize
from raw_preprocessing import preprocess_raw

# === USER PARAMETERS ===
RAW_PATH    = "testimages/img146.jpg"
MODEL_PATH  = "complex_unet_enhancedv5.keras"
OUTPUT_CSV  = "layer_measurements.csv"

SIZE        = 496      # your model input size
L_PX        = 50       # ← measure (in pixels) your 200 µm scale bar
SCALE_UM    = 200.0    # µm corresponding to L_PX
N_CLASSES   = 12
CLASS_NAMES = {
    0:"Background", 1:"ILM", 2:"RNFL", 3:"GCL", 4:"IPL",
    5:"INL", 6:"OPL", 7:"ELM", 8:"PR1", 9:"PR2", 10:"RPE", 11:"BM"
}

# 1) Load & preprocess the raw OCT → (H,W,1) in [0,1]
raw_3d = preprocess_raw(RAW_PATH, target_size=SIZE, normalize=True)
raw    = raw_3d[...,0]  # (H, W)

# 2) Load the model and predict the integer mask
model = load_model(MODEL_PATH, compile=False)
X     = normalize(raw_3d[np.newaxis,...], axis=1)  # (1,H,W,1)
pred  = model.predict(X)[0]                        # (H,W,12)
mask  = np.argmax(pred, axis=-1).astype(np.uint8)  # (H,W)
print("Unique classes in mask:", np.unique(mask))

# 3) Compute calibration
um_per_px = SCALE_UM / L_PX
print(f"Calibration: {um_per_px:.2f} µm/px")

# 4) Measure each layer
h, w = mask.shape
rows = np.arange(h)
records = []

for cls in range(1, N_CLASSES):
    ys_per_col = [rows[mask[:,c]==cls] for c in range(w)]
    # thickness in px where the layer exists
    th_px = [y[-1]-y[0]+1 for y in ys_per_col if y.size]
    if not th_px:
        continue
    th_um = np.array(th_px)*um_per_px
    refl  = raw[mask==cls]
    records.append({
        "Layer":            CLASS_NAMES[cls],
        "ColumnsMeasured":  len(th_px),
        "MeanThickness_µm": np.mean(th_um),
        "SDThickness_µm":   np.std(th_um),
        "MeanReflectivity": np.mean(refl),
        "SDReflectivity":   np.std(refl),
    })

# 5) Build DataFrame and save
df = pd.DataFrame.from_records(
    records,
    columns=["Layer","ColumnsMeasured",
             "MeanThickness_µm","SDThickness_µm",
             "MeanReflectivity","SDReflectivity"]
)
print(df.to_string(index=False))

df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Measurements saved to {OUTPUT_CSV}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Unique classes in mask: [ 0  1  2  3  4  5  6  7  8  9 10 11]
Calibration: 4.00 µm/px
Layer  ColumnsMeasured  MeanThickness_µm  SDThickness_µm  MeanReflectivity  SDReflectivity
  ILM              456         31.271930       95.887341          0.499103        0.290956
 RNFL              496         36.798387       15.317331          0.534952        0.149324
  GCL              492         30.512195       11.829319          0.522672        0.090561
  IPL              477         32.637317       10.936663          0.464623        0.099045
  INL              490         29.844898        9.256020          0.487772        0.092531
  OPL              496         70.572581       14.368241          0.327387        0.094064
  ELM              496         23.588710        2.212545          0.387597        0.083372
  PR1              496         20.104839        2.346303          0.739242        0.142168
  PR2              496         24.008065        2.879281   